In [21]:
import numpy as np 
import pandas as pd 


In [22]:
df = pd.read_csv('movies_dataset.csv')
df.head()

,id,title,overview,genre
0,1560520,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...","['Animation', 'Action', 'Adventure', 'Drama']"
1,980431,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...","['Animation', 'Adventure', 'Fantasy', 'Action']"
2,1007757,Swapped,"A small woodland creature and a majestic bird,...","['Adventure', 'Animation', 'Family', 'Fantasy']"
3,1632181,Accidental Partners,Two women discover they were both scammed by t...,"['Comedy', 'Romance']"
4,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,"['Animation', 'Action', 'Fantasy']"


Text preprocessing on the overview

## LowerCasing

In [23]:
df['overview'][0].lower()

'arkham asylum has been destroyed, and all its inmates have been unleashed upon gotham city. as batman races to round up some of his greatest enemies, he is pushed to his physical and mental limits and into a final confrontation with a new threat: the man called bane!'

In [24]:
df['overview'] = df['overview'].str.lower()


In [25]:
df.head()

,id,title,overview,genre
0,1560520,Batman: Knightfall Part 1: Knightfall,"arkham asylum has been destroyed, and all its ...","['Animation', 'Action', 'Adventure', 'Drama']"
1,980431,Avatar Aang: The Last Airbender,"avatar aang, the world's last airbender, learn...","['Animation', 'Adventure', 'Fantasy', 'Action']"
2,1007757,Swapped,"a small woodland creature and a majestic bird,...","['Adventure', 'Animation', 'Family', 'Fantasy']"
3,1632181,Accidental Partners,two women discover they were both scammed by t...,"['Comedy', 'Romance']"
4,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,the demon slayer corps are drawn into the infi...,"['Animation', 'Action', 'Fantasy']"


In [26]:
df['overview'][3]

'two women discover they were both scammed by the same man (who also got them pregnant). they form an alliance to take revenge.'

## Removing html if present

In [27]:
import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'',str(text))

df['overview'] = df['overview'].apply(remove_html_tags)

Removing url if present

In [28]:
def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', str(text))

df['overview'] = df['overview'].apply(remove_url)


Removing punctuation

In [29]:
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [30]:
exclude = string.punctuation

In [31]:
df['overview'][1]


"avatar aang, the world's last airbender, learns of an ancient power that could save his culture from extinction. with the help of his friends, he embarks on a global quest to find it before it falls into the wrong hands and threatens to upend the peace they sacrificed everything to achieve."

In [32]:
def remove_punc_fast(text):
    return text.translate(str.maketrans('', '', exclude))

df['overview'] = df['overview'].apply(remove_punc_fast)


In [33]:
df['overview'][1]

'avatar aang the worlds last airbender learns of an ancient power that could save his culture from extinction with the help of his friends he embarks on a global quest to find it before it falls into the wrong hands and threatens to upend the peace they sacrificed everything to achieve'

## ChatWord Conversion 

Not needed here but might be needed mainly on the messages kind of data

## MisSpelled Treatment

In [34]:
from textblob import TextBlob

def fix_spelling(text):
    textBlb = TextBlob(text)

    return textBlb.correct().string

fix_spelling('ceertain')


'certain'

In [ ]:
df['overview'] = df['overview'].apply(fix_spelling)


## Removing Stop words 

In [37]:
from nltk.corpus import stopwords

In [39]:
# stopwords.words('english')

In [40]:
def remove_stopwords(text):
    new_text = []
    
    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')
        else:
            new_text.append(word)
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [41]:
df['overview'] = df['overview'].apply(remove_stopwords)


In [42]:
df.sample()

,id,title,overview,genre
3281,547016,The Old Guard,four undying warriors whove secretly protected...,"['Action', 'Fantasy']"


## Remove Emojis / replace emoji with meaning

Not needed here

## Tokenizations 

using spacy for tokenization

In [43]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [51]:
def tokenize(text):
    docs = nlp(text)
    tokens =[]
    for token in docs:
        tokens.append(token)
    return tokens

# print(type(tokenize("Hello what's my name ?")))
print(tokenize("Hello what's my name ?"))

[Hello, what, 's, my, name, ?]


In [52]:
df['tokens'] = df['overview'].apply(tokenize)


In [65]:
df.sample()
type(df['tokens'][0])

list

## Stemming 


In [56]:
from nltk.stem.porter import PorterStemmer # SnowballStemmer for the Other languages this works well for the english
ps = PorterStemmer()


In [66]:
# def stem_words(tokens):
#     new_list =[]
#     for token in tokens:
#         new_list.append(ps.stem(token))
#     return new_list
        

# Better code to avoid error due to numbers and empty values 

def stem_words(tokens):
    # 1. Guard against rows that aren't lists (like NaN or float)
    if not isinstance(tokens, (list, tuple)):
        return []

    # 2. Stringify tokens and skip empty/null items to prevent the .lower() bug
    return [
        ps.stem(str(token))
        for token in tokens
        if token is not None and str(token).strip() != ""
    ]


# Apply the safe function
df["tokens"] = df["tokens"].apply(stem_words)

    # return " ".join([ps.stem(word) for word in text.split()])
test = [ "do ", "does ", "did", "doing" ]
stem_words(test)

['do ', 'does ', 'did', 'do']

In [67]:
df['tokens'] = df['tokens'].apply(stem_words)


In [68]:
df.sample()

,id,title,overview,genre,tokens
3580,412202,Handsome Devil,musicmad 16yearold outcast rugbymad boardin...,"['Drama', 'Comedy']","[musicmad, 16yearold, outcast, rugbymad, board..."


## Lemmitizations 

Using WordNet to find the lemma (a word that always exit in language)

##### STemming vs Lemmatization
The best approach depends entirely on whether you want maximum speed (Stemming) or maximum linguistic accuracy (Lemmatization). You rarely want to do both to the same word, as stemming a lemmatized word (or vice versa) ruins the text alignment.
For modern text processing pipelines, the gold standard is to use lemmatization via Spacy because it looks at the whole sentence structure to understand context (e.g., distinguishing between "building" a house vs. a tall "building").
Here is the breakdown of the best tools and implementations for both methods.
------------------------------
###### Method 1: The Best Way to Lemmatize (Spacy - Accurate & Modern)
[spaCy](https://spacy.io/) is significantly faster and more accurate than older tools like NLTK's WordNetLemmatizer because it automatically detects parts of speech (POS tagging).

import spacy
###### Load the small, optimized English modelnlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize_text(text):
    # Process the text through spaCy
    doc = nlp(text)
    # Extract lemmas for every token
    return [token.lemma_ for token in doc]

###### Example
print(
    lemmatize_text("The mice were running quickly and saw two buildings.")
)# Output: ['the', 'mouse', 'be', 'run', 'quickly', 'and', 'see', 'two', 'building', '.']

Note: Notice how "mice" correctly becomes "mouse" and "were" becomes "be".
------------------------------
###### Method 2: The Best Way to Stem (Snowball / Porter2 - Fast & Light)
If you are building an information retrieval engine (like a search bar index) and just need crude, lightning-fast root matching, use [NLTK's Snowball Stemmer](https://www.nltk.org/api/nltk.stem.snowball.html).

from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("english")

def stem_tokens(tokens):
    # Cast to str to prevent the int/float bugs encountered earlier
    return [stemmer.stem(str(token)) for token in tokens]

##### Example
print(stem_tokens(["mice", "running", "buildings"]))# Output: ['mice', 'run', 'build']

Note: Stemmers use rigid rules, so "mice" stays "mice", and "buildings" is crudely chopped down to "build".
------------------------------
###### Direct Comparison: Stemming vs. Lemmatization

| Feature | Stemming (Snowball) | Lemmatization (spaCy) |
|---|---|---|
| How it works | Chops off suffixes using brute-force rules. | Uses a dictionary and context to find the real dictionary root. |
| Speed | 🚀 Blazing Fast (lightweight math calculations). | 🐢 Slower (requires loading an NLP language model). |
| Output Type | Often yields non-words ("studying" → "studi"). | Always yields real, readable words ("studying" → "study"). |
| Context Aware | No. Treats every word completely isolated. | Yes. Knows if a word is a verb, noun, or adjective. |

##### Summary Recommendation

* 
* Choose Lemmatization (spaCy) if you are building an AI chatbot, doing sentiment analysis, or feeding text into an Embedding/Transformer model where real word meaning matters.
* Choose Stemming (Snowball) if you have a massive dataset of millions of documents and you just need a lightweight, fast algorithm for basic keyword matching.
* 

Which specific task are you trying to accomplish with this data—are you preparing text for a machine learning model, building a search filter, or running sentiment analysis?



Using WordNet from nltk

In [71]:
import nltk
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()
def lemmatize(tokens):
    
    return [wordnet_lemmatizer.lemmatize(word,pos='v') for word in tokens]

test = ['the', 'mice', 'is', 'running', 'quickly', 'and', 'sees', 'two', 'building', '.']
lemmatize(test)

['the', 'mice', 'be', 'run', 'quickly', 'and', 'see', 'two', 'build', '.']

In [72]:
df['tokens'] = df['tokens'].apply(lemmatize)


In [73]:
df.sample()

,id,title,overview,genre,tokens
4033,2140,Kiss of the Dragon,liu jian elite chinese police officer comes ...,"['Action', 'Crime', 'Thriller']","[liu, jian, elit, chine, polic, offic, come, p..."


Using Spacy

Spcy is considered a better one 

spaCy is significantly faster and more accurate than older tools like NLTK's WordNetLemmatizer because it automatically detects parts of speech (POS tagging).

In [69]:
import spacy

# Load the small, optimized English model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])


def lemmatize_text(text):
    # Process the text through spaCy
    doc = nlp(text)
    # Extract lemmas for every token
    return [token.lemma_ for token in doc]


# test
print(
    lemmatize_text("The mice were running quickly and saw two buildings.")
)



['the', 'mouse', 'be', 'run', 'quickly', 'and', 'see', 'two', 'building', '.']
